In [1]:
import os
os.chdir('../new_scripts')
import pandas as pd
from deltalake import DeltaTable
from src.utils import WMAPE, wmape
import matplotlib.pyplot as plt
from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoLSTM
import datetime
import joblib
from src.utils import setLog
import mlflow
import mlflow.pyfunc
import psutil
import platform

In [2]:
df = DeltaTable('deltalake').to_pandas()
df = df.sort_values(by=['unique_id', 'ds']).reset_index(drop=True)

In [3]:
df = DeltaTable('deltalake').to_pandas()
df = df.sort_values(by=['unique_id', 'ds']).reset_index(drop=True)

df = df.loc[df['ds'] > '2023-01-01']

# # Separando dados de treinamento e testes
train = df.loc[df['ds'] < '2024-09-01']
valid = df.loc[(df['ds'] >= '2024-09-01') & (df['ds'] < '2025-01-31')]

h = train['ds'].nunique()

models = [AutoLSTM(h=h, 
                num_samples=30, 
                loss=WMAPE())]

model = NeuralForecast(models=models, freq='D')

In [4]:
initial_config = models[0].config

In [5]:
initial_config

{'h': 417,
 'encoder_hidden_size': <ray.tune.search.sample.Categorical at 0x7f7ab72d4790>,
 'encoder_n_layers': <ray.tune.search.sample.Integer at 0x7f7ab72f1810>,
 'context_size': <ray.tune.search.sample.Categorical at 0x7f7ab72f3010>,
 'decoder_hidden_size': <ray.tune.search.sample.Categorical at 0x7f7c1bd61690>,
 'learning_rate': <ray.tune.search.sample.Float at 0x7f7ab72f3090>,
 'max_steps': <ray.tune.search.sample.Categorical at 0x7f7ab72f30d0>,
 'batch_size': <ray.tune.search.sample.Categorical at 0x7f7ab72f3150>,
 'loss': WMAPE(),
 'random_seed': <ray.tune.search.sample.Integer at 0x7f7ab72f31d0>,
 'input_size': <ray.tune.search.sample.Categorical at 0x7f7ab72f3250>,
 'inference_input_size': <ray.tune.search.sample.Categorical at 0x7f7ab72f32d0>,
 'valid_loss': WMAPE()}

In [ ]:
model.fit(train, val_size=30)

(_train_tune pid=90422) /home/thales/postech/phase4/mle-tech-challenge-4/.venv/lib/python3.11/site-packages/ray/tune/integration/pytorch_lightning.py:198: `ray.tune.integration.pytorch_lightning.TuneReportCallback` is deprecated. Use `ray.tune.integration.pytorch_lightning.TuneReportCheckpointCallback` instead.
(_train_tune pid=90422) Seed set to 5
(_train_tune pid=90422) GPU available: True (cuda), used: True
(_train_tune pid=90422) TPU available: False, using: 0 TPU cores
(_train_tune pid=90422) HPU available: False, using: 0 HPUs
(_train_tune pid=90422) You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
(_train_tune pid=90422) LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: 

Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]


(_train_tune pid=90422) 
(_train_tune pid=90422)   | Name            | Type          | Params | Mode 
(_train_tune pid=90422) ----------------------------------------------------------
(_train_tune pid=90422) 0 | loss            | WMAPE         | 0      | train
(_train_tune pid=90422) 1 | padder          | ConstantPad1d | 0      | train
(_train_tune pid=90422) 2 | scaler          | TemporalNorm  | 0      | train
(_train_tune pid=90422) 3 | hist_encoder    | LSTM          | 122 K  | train
(_train_tune pid=90422) 4 | context_adapter | Linear        | 2.1 M  | train
(_train_tune pid=90422) 5 | mlp_decoder     | MLP           | 26.6 K | train
(_train_tune pid=90422) ----------------------------------------------------------
(_train_tune pid=90422) 2.3 M     Trainable params
(_train_tune pid=90422) 0         Non-trainable params
(_train_tune pid=90422) 2.3 M     Total params
(_train_tune pid=90422) 9.018     Total estimated model params size (MB)
(_train_tune pid=90422) 11        Modules in

Epoch 1: 100%|██████████| 1/1 [00:01<00:00,  0.73it/s, v_num=0, train_loss_step=1.010, train_loss_epoch=1.010]
